# 第5章　データの扱い（`Dataset` / `DataLoader`）

実データは大きく、丸ごとメモリに乗せられません。PyTorch は **`Dataset`（1件の取り出し方）** と
**`DataLoader`（ミニバッチ・シャッフル・並列読込）** で効率よく供給します。

この章のゴール：自作 `Dataset` を書け、`DataLoader` でバッチを回せ、`torchvision` の画像データを読める。

> **このノートの使い方**
> - 上から順にセルを実行（Colab/Jupyter ともに `Shift + Enter`）。
> - コードは**少し書き換えて壊して直す**のが一番伸びます。各章末に演習があります。
> - GPU は不要な章が多いです。重い章（CNN）では使い方を案内します。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 5-1. ミニバッチとは？
全データを一度に使わず、**少数（例：32件）ずつ**に分けて学習します。これがミニバッチ。
1回のループで1バッチ処理。全データを1周することを **1 epoch** と呼びます。

## 5-2. 自作 `Dataset`
`Dataset` は2つのメソッドを実装するだけ：
- `__len__`：データ件数
- `__getitem__(i)`：i 番目の (入力, ラベル) を返す

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, n=100):
        self.X = torch.randn(n, 3)               # 特徴量
        self.y = (self.X.sum(dim=1) > 0).long()  # 合計が正なら1
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

ds = MyDataset(100)
print("件数:", len(ds))
print("0番目:", ds[0])

## 5-3. `DataLoader` でバッチに分ける
`batch_size` でバッチの大きさ、`shuffle=True` で毎 epoch 順番をシャッフル（学習では基本 True）。

In [ ]:
loader = DataLoader(ds, batch_size=16, shuffle=True)

for xb, yb in loader:
    print("バッチ:", xb.shape, yb.shape)   # (16, 3), (16,)
    break   # 最初の1バッチだけ表示

print("1 epoch のバッチ数:", len(loader))   # 100/16 = 7 バッチ

## 5-4. 画像データ：`torchvision` + `transforms`

`torchvision` には MNIST 等の定番データセットと、前処理 `transforms` が入っています。
- `transforms.ToTensor()`：画像を `[0,1]` の Tensor に変換し、形を `(C, H, W)` に整える。
- `Normalize`：平均・標準偏差で正規化（学習が安定）。
- `Compose`：複数の前処理を順に適用。

> 初回はダウンロードが走ります（数十MB、ネット接続が必要）。Colab なら数秒。

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),                 # (1,28,28), 値0..1
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST の平均/標準偏差
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print("学習データ件数:", len(train_ds))
img, label = train_ds[0]
print("1枚の形:", img.shape, " ラベル:", label)   # (1,28,28), 5

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1000, shuffle=False)

xb, yb = next(iter(train_loader))
print("バッチ:", xb.shape, yb.shape)   # (64,1,28,28), (64,)

### 画像を表示してみる

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, i in zip(axes, range(6)):
    img, label = train_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")   # (1,28,28)->(28,28)
    ax.set_title(str(label)); ax.axis("off")
plt.show()

## 5-5. 学習用/検証用に分ける：`random_split`
手元の学習データをさらに train / val に分けて、過学習をチェックします。

In [ ]:
from torch.utils.data import random_split
n_val = 10000
n_train = len(train_ds) - n_val
tr, va = random_split(train_ds, [n_train, n_val])
print("train:", len(tr), " val:", len(va))

## 演習 5
1. `MyDataset` の `batch_size` を 8 / 64 に変え、1 epoch のバッチ数の変化を確認しよう。
2. MNIST の画像をもっと多く（例：16枚）並べて表示してみよう。
3. `transforms` に `transforms.RandomRotation(15)` を足すと画像がどう変わる？（データ拡張の入口）

In [ ]:
# ここに自分のコードを書いて実行してみよう
